In [1]:
import pandas as pd

df = pd.read_csv("hf://datasets/opensporks/resumes/Resume/Resume.csv")

In [2]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [ ]:
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor

# 1. Initialize SkillNER (Do this outside the loop so it only loads once)
print("Loading NLP model...")
nlp = spacy.load("en_core_web_lg")
skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)

# --- Mocking your DataFrame for demonstration ---
data = {
    'ID': [0, 1],
    'Resume_str': [
        "HR ADMINISTRATOR/MARKETING ASSOCIATE. Experienced in leadership, employee relations, recruitment, and Microsoft Excel.",
        "Data Scientist. Proficient in Python, Machine Learning, SQL, and critical thinking."
    ],
    'Resume_html': ["<div class='fontsize...'>HR</div>", "<div class='fontsize...'>IT</div>"],
    'Category': ["HR", "Data Science"]
}
df = pd.DataFrame(data)
# ------------------------------------------------

# 2. Define a safe extraction function for Pandas
def extract_skills_from_row(text):
    """Safely extracts skills from a single text string."""
    # Safety check: skip nulls or empty strings
    if pd.isna(text) or not str(text).strip():
        return []
    
    try:
        # Run SkillNER
        annotations = skill_extractor.annotate(str(text))
        extracted_skills = []
        
        # Get exact matches
        for match in annotations.get('results', {}).get('full_matches', []):
            extracted_skills.append(match['doc_node_value'])
            
        # Get multi-word matches
        for match in annotations.get('results', {}).get('ngram_scored', []):
            extracted_skills.append(match['doc_node_value'])
            
        # Return a clean list of unique skills
        return list(set(extracted_skills))
        
    except Exception as e:
        # If a specific row fails (e.g., text is too long for spaCy), catch it and keep going
        print(f"Error extracting skills: {e}")
        return []

# 3. Apply the function to create a new column
print("Extracting skills across the DataFrame. This might take a moment depending on dataset size...")

# Apply the function to the 'Resume_str' column
df['Extracted_Skills'] = df['Resume_str'].apply(extract_skills_from_row)

# 4. View the results!
print("\n--- Pipeline Complete ---")
print(df[['ID', 'Category', 'Extracted_Skills']])


Loading NLP model...
loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...
Extracting skills across the DataFrame. This might take a moment depending on dataset size...

--- Pipeline Complete ---
   ID      Category                                   Extracted_Skills
0   0            HR   [microsoft excel, employee relation, leadership]
1   1  Data Science  [python, critical thinking, machine learning, ...


In [ ]:
# try on df
df = pd.read_csv("hf://datasets/opensporks/resumes/Resume/Resume.csv")
# 2. Define a safe extraction function for Pandas
def extract_skills_from_row(text):
    """Safely extracts skills from a single text string."""
    # Safety check: skip nulls or empty strings
    if pd.isna(text) or not str(text).strip():
        return []
    
    try:
        # Run SkillNER
        annotations = skill_extractor.annotate(str(text))
        extracted_skills = []
        
        # Get exact matches
        for match in annotations.get('results', {}).get('full_matches', []):
            extracted_skills.append(match['doc_node_value'])
            
        # Get multi-word matches
        for match in annotations.get('results', {}).get('ngram_scored', []):
            extracted_skills.append(match['doc_node_value'])
            
        # Return a clean list of unique skills
        return list(set(extracted_skills))
        
    except Exception as e:
        # If a specific row fails (e.g., text is too long for spaCy), catch it and keep going
        print(f"Error extracting skills: {e}")
        return []

# 3. Apply the function to create a new column
print("Extracting skills across the DataFrame. This might take a moment depending on dataset size...")

# Apply the function to the 'Resume_str' column
df['Extracted_Skills'] = df['Resume_str'].apply(extract_skills_from_row)

# 4. View the results!
print("\n--- Pipeline Complete ---")
print(df[['ID', 'Category', 'Extracted_Skills']])


Extracting skills across the DataFrame. This might take a moment depending on dataset size...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)
